## Concept 8 — Self-Correcting Agent

### What is it?
A Self-Correcting Agent adds a **Critic LLM** after the initial answer.
The Critic evaluates whether the answer is correct and complete.
If it fails, the agent retries with the Critic's feedback — automatically.

### Pipeline
```
Query
  → Agent produces answer (Attempt 1)
  → Critic: PASS → return answer immediately
       or
  → Critic: FAIL 'missing X, Y' → retry with feedback (Attempt 2)
  → Critic: PASS → return answer
       or
  → Critic: FAIL again → retry (Attempt 3, max)
  → return best answer
```

### Why a Separate Critic?
The answering LLM tends to think its own answer is correct.
A **separate Critic call** is more objective — it evaluates without bias toward the original answer.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm      = get_llm(temperature=0.0)
tool_map = {t.name: t for t in TOOLS}
print(f'LLM ready | Tools: {list(tool_map.keys())}')

### Step 1 — The Base Agent
Standard tool-calling agent — same as Concept 1. This is what we will validate.

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)

def base_agent(query: str, feedback: str = '') -> str:
    prompt = query
    if feedback:
        prompt = f'{query}\n\n[Previous attempt was incorrect. Feedback: {feedback}. Please try again carefully.]'

    messages = [HumanMessage(content=prompt)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            print(f'  [Tool] {call["name"]}({call["args"]}) → {result}')
            messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        final = llm_with_tools.invoke(messages)
        return final.content

    return response.content

# Test base agent
print('Base agent test:')
ans = base_agent('What is 144 divided by 12?')
print(f'Answer: {ans}')

### Step 2 — The Critic
The Critic is an independent LLM call that scores the answer PASS or FAIL.

In [ ]:
critic_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a strict quality evaluator for an AI assistant.\n'
     'Your job: evaluate whether the answer fully and correctly addresses the query.\n\n'
     'Evaluation criteria:\n'
     '1. Does the answer address ALL parts of the query?\n'
     '2. Are numerical results correct (no hallucinated math)?\n'
     '3. Is the answer specific, not vague?\n'
     '4. Were appropriate tools used (math → calculate, docs → search_docs, weather → get_weather)?\n\n'
     'Reply format:\n'
     'VERDICT: PASS  (if all criteria met)\n'
     'VERDICT: FAIL\nREASON: [specific issue]  (if any criterion failed)'),
    ('human',
     'Query: {query}\n\n'
     'Agent Answer: {answer}\n\n'
     'Evaluate:'),
])
critic_chain = critic_prompt | llm | StrOutputParser()

def critique(query: str, answer: str) -> tuple:
    evaluation = critic_chain.invoke({'query': query, 'answer': answer})
    passed     = 'VERDICT: PASS' in evaluation
    feedback   = ''
    if not passed:
        lines    = evaluation.split('\n')
        reason   = [l for l in lines if 'REASON' in l]
        feedback = reason[0].replace('REASON:', '').strip() if reason else evaluation
    return passed, feedback, evaluation

# Test critic on a good answer
ans = base_agent('What is 144 divided by 12?')
passed, feedback, evaluation = critique('What is 144 divided by 12?', ans)
print(f'Answer:   {ans}')
print(f'Verdict:  {"PASS" if passed else "FAIL"}')
print(f'Feedback: {feedback}')

### Step 3 — Simulate a Failure Case
Inject a deliberately bad answer to see the Critic catch it.

In [ ]:
bad_answer = 'The answer is 15.'  # wrong answer to 144/12
passed, feedback, evaluation = critique('What is 144 divided by 12?', bad_answer)
print(f'Bad answer: "{bad_answer}"')
print(f'Verdict:    {"PASS" if passed else "FAIL"}')
print(f'Evaluation:\n{evaluation}')

### Step 4 — Full Self-Correcting Loop
Agent answers → Critic evaluates → retry with feedback if FAIL → max retries → best answer.

In [ ]:
MAX_RETRIES = 3

def self_correcting_agent(query: str) -> str:
    feedback = ''

    for attempt in range(1, MAX_RETRIES + 1):
        print(f'\n[Attempt {attempt}] Generating answer...')
        answer = base_agent(query, feedback)
        print(f'  Answer: {answer[:150]}...' if len(answer) > 150 else f'  Answer: {answer}')

        passed, feedback, _ = critique(query, answer)
        print(f'  Critic: {"PASS ✓" if passed else f"FAIL ✗ — {feedback}"}')

        if passed:
            print(f'  → Returning answer after {attempt} attempt(s)')
            return answer

    print(f'  → Max retries reached. Returning last answer.')
    return answer

# Demo
print('=== Self-Correcting Agent Demo ===')
result = self_correcting_agent('What is the weather in Hyderabad?')
print(f'\n=== FINAL ANSWER ===\n{result}')

### Bonus — Multi-Part Query (Critic Checks All Parts)
The Critic ensures BOTH parts of a multi-part query are answered.

In [ ]:
multi_q = 'Search docs for RAG and also calculate 25 multiplied by 4'
print(f'Query: {multi_q}\n')
result = self_correcting_agent(multi_q)
print(f'\nFinal Answer: {result}')

### Full Function

In [ ]:
run_queries(self_correcting_agent, label='Concept 8 — Self-Correcting Agent')